# 5. Neural Network Model Development


## 5.1 Problem Definition and Modeling Strategy


The objective of this project is to develop a neural network model for predicting daily sales of Rossmann stores.

The task is formulated as a supervised regression problem.

Given the engineered feature vector:

$$
X_i =
(x_1,x_2,...,x_p)
$$

the model aims to predict the corresponding daily sales value:

$$
y_i = Sales_i
$$


The neural network learns a nonlinear mapping function:

$$
\hat{y_i}=f(X_i;\theta)
$$

where:

- \(X_i\) represents the input feature vector.
- \(y_i\) represents the observed sales value.
- \(\hat{y_i}\) represents the predicted sales.
- \(\theta\) represents the trainable parameters of the neural network.


## Input Features

The model input is based on the final feature representation generated in the feature engineering stage.

The input contains the encoded features produced from four information groups:

1. Historical demand features.

2. Temporal features.

3. Promotion features.

4. Store characteristics.


Same-day Customers information is excluded because it is unavailable in the test dataset.

This prevents information leakage and ensures that the model can be applied to future prediction scenarios.


## Prediction Target

The prediction target is:

$$
Target = Sales
$$


The model directly predicts daily sales values.

Since Sales is a continuous variable, this task is treated as a regression problem.


## Loss Function

The neural network is optimized by minimizing Mean Squared Error (MSE):

$$
MSE =
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y_i})^2
$$


MSE is selected because large prediction errors should receive stronger penalties in sales forecasting.


## Evaluation Metrics

Model performance will be evaluated using:

### Root Mean Squared Error (RMSE)

$$
RMSE=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y_i})^2
}
$$


### Mean Absolute Error (MAE)

$$
MAE=
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y_i}|
$$


RMSE emphasizes large prediction errors, while MAE represents the average prediction deviation.

Together, they provide a comprehensive evaluation of forecasting performance.


## Modeling Strategy

The neural network model is designed to learn nonlinear relationships among:

$$
Historical\ Demand
+
Temporal\ Pattern
+
Promotion\ Effect
+
Store\ Characteristics
\rightarrow
Sales
$$


The following sections will prepare the data, construct the neural network architecture, train the model, and evaluate prediction performance.

# 5.2 Time-based Train Validation Split


The Rossmann sales forecasting task is a future prediction problem.

Therefore, random train-validation splitting is inappropriate because it may introduce temporal leakage.

Instead, a time-based split strategy is applied.

The training and validation datasets are divided according to chronological order:


$$
Training\ Period
<
Validation\ Period
<
Testing\ Period
$$


The validation set is selected from the latest period of the training data to simulate future prediction.


The split strategy is:

- Training period:

$$
2013-01-01 \sim 2015-06-30
$$


- Validation period:

$$
2015-07-01 \sim 2015-07-31
$$


- Testing period:

$$
2015-08-01 \sim 2015-09-17
$$


This strategy ensures that validation performance reflects the model's ability to generalize to unseen future sales data.

In [ ]:
# 5.1-5.2 Load the exported feature tables and split by date
# This notebook does not reuse objects from 04_features.ipynb.

from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().parent

TRAIN_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_feature_final.csv"
)

TEST_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "test_feature_final.csv"
)


if not TRAIN_FEATURE_PATH.exists() or not TEST_FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Run 04_features.ipynb through the export step before modeling."
    )


train_feature = pd.read_csv(
    TRAIN_FEATURE_PATH,
    parse_dates=["Date"]
)

test_feature = pd.read_csv(
    TEST_FEATURE_PATH,
    parse_dates=["Date"]
)


if "Customers" in train_feature.columns or "Customers" in test_feature.columns:
    raise ValueError(
        "Same-day Customers must not enter the modeling dataset."
    )


if set(train_feature.columns) - {"Sales"} != set(test_feature.columns):
    raise ValueError(
        "Train and test feature columns are not aligned."
    )


# =====================================================
# Time-based split. Do not use random train_test_split.
# =====================================================

train_end_date = pd.Timestamp("2015-06-30")

validation_start_date = pd.Timestamp("2015-07-01")

validation_end_date = pd.Timestamp("2015-07-31")



# =====================================================
# 2. Split feature dataset
# =====================================================


train_data = train_feature[
    train_feature["Date"] <= train_end_date
].copy()



validation_data = train_feature[
    (train_feature["Date"] >= validation_start_date)
    &
    (train_feature["Date"] <= validation_end_date)
].copy()



print(
    "Training period:",
    train_data["Date"].min(),
    "to",
    train_data["Date"].max()
)


print(
    "Validation period:",
    validation_data["Date"].min(),
    "to",
    validation_data["Date"].max()
)



# =====================================================
# 3. Separate features and target
# =====================================================


target_column = "Sales"

drop_columns = [
    "Sales",
    "Date",
    "Customers"
]

feature_columns = [
    column for column in train_feature.columns
    if column not in drop_columns
]

X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_val = validation_data[feature_columns]
y_val = validation_data[target_column]

X_test = test_feature[feature_columns]


# =====================================================
# 4. Check dataset shapes
# =====================================================

split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Samples": [
        X_train.shape[0],
        X_val.shape[0],
        X_test.shape[0]
    ],
    "Features": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],
    "Date range": [
        f"{train_data['Date'].min():%Y-%m-%d} to {train_data['Date'].max():%Y-%m-%d}",
        f"{validation_data['Date'].min():%Y-%m-%d} to {validation_data['Date'].max():%Y-%m-%d}",
        f"{test_feature['Date'].min():%Y-%m-%d} to {test_feature['Date'].max():%Y-%m-%d}"
    ]
})

display(split_summary)

## 5.3 Feature Scaling

Neural network training is sensitive to feature scale. Scaling is therefore fitted only after the time-based split, and only on the training period from 2013-01-01 to 2015-06-30.

The validation period and the future test period are transformed with the training-period statistics. They are not used to estimate means, standard deviations, category levels, or imputation values.

Historical features can be missing at the beginning of a store series, or when a test date looks back into dates whose sales are not yet known. Those missing numeric values are replaced by the training-period median before standardization. This imputation is part of the modeling interface, not a new feature.

Categorical variables are one-hot encoded. Unknown categories in later periods are ignored rather than rejected.


In [ ]:
# 5.3 Feature scaling
# Fit only on the training period, then transform validation and test.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


categorical_features = (
    X_train
    .select_dtypes(include=["object", "string"])
    .columns
    .tolist()
)

numerical_features = [
    column for column in X_train.columns
    if column not in categorical_features
]


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

model_preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)


X_train_model = model_preprocessor.fit_transform(X_train)
X_val_model = model_preprocessor.transform(X_val)
X_test_model = model_preprocessor.transform(X_test)


if hasattr(X_train_model, "toarray"):
    X_train_model = X_train_model.toarray()
    X_val_model = X_val_model.toarray()
    X_test_model = X_test_model.toarray()


model_feature_names = model_preprocessor.get_feature_names_out()

scaling_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Samples": [
        X_train_model.shape[0],
        X_val_model.shape[0],
        X_test_model.shape[0]
    ],
    "Features": [
        X_train_model.shape[1],
        X_val_model.shape[1],
        X_test_model.shape[1]
    ]
})

display(scaling_summary)

print("Scaled model input features:", len(model_feature_names))
print("Scaler fitted on training period only: True")


## 5.4 Neural Network Modeling

The scaled training matrix is used to fit a multilayer perceptron regressor. The validation period is not used for fitting the scaler or the network weights.

The network minimizes squared error. Validation performance is reported with RMSE and MAE so that large errors and average errors can be read separately.

If `Open` is known to be 0, the corresponding prediction is set to 0 after inference. Closed stores do not generate sales, and this rule uses only information available at prediction time.


In [ ]:
# 5.4 Neural network modeling

import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neural_network import MLPRegressor


sales_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=False,
    random_state=42
)

sales_model.fit(X_train_model, y_train)


validation_prediction = sales_model.predict(X_val_model)

if "Open" in validation_data.columns:
    closed_validation = validation_data["Open"].to_numpy() == 0
    validation_prediction = validation_prediction.copy()
    validation_prediction[closed_validation] = 0


validation_metrics = pd.DataFrame({
    "Dataset": ["Validation"],
    "RMSE": [
        root_mean_squared_error(y_val, validation_prediction)
    ],
    "MAE": [
        mean_absolute_error(y_val, validation_prediction)
    ]
})

display(validation_metrics)


test_prediction = sales_model.predict(X_test_model)

if "Open" in test_feature.columns:
    closed_test = test_feature["Open"].to_numpy() == 0
    test_prediction = test_prediction.copy()
    test_prediction[closed_test] = 0

test_prediction = np.clip(test_prediction, 0, None)

print("Test predictions:", test_prediction.shape)
print("Neural network training iterations:", sales_model.n_iter_)
